In [1]:
using L2Pipeline
using GLMakie
using HDF5
using Formatting

H5SAVE = abspath("../Data/processing_steps.hdf5")
println("Saving data to: \n$(H5SAVE)")

h = h5open(H5SAVE,"w")
close(h)

Saving data to: 
c:\Users\zvig\.julia\dev\L2Pipeline.jl\Data\processing_steps.hdf5


In [2]:
caldata_path = "C:/Lunar_Imagery_Data/M3_data/020644_Caldata/"

"C:/Lunar_Imagery_Data/M3_data/020644_Caldata/"

In [3]:
caldata = caldata_from_url(caldata_path,"020644")
println(fieldnames(typeof(caldata)))

(:wvl, :rdn, :obs, :illum, :solspec, :statpol, :falpha, :current_step)


In [4]:
rem_solspec!(caldata)

h5open(H5SAVE,"cw") do h
    h["solspec_removed"] = caldata.current_step
    return nothing
end

I/F Correction Complete!


In [5]:
statistical_polish!(caldata)

h5open(H5SAVE,"cw") do h
    h["IoverF_statpol"] = caldata.current_step
    return nothing
end

Cold instrument polishing applied...Statistical Polishing Complete!


In [6]:
d = clark_etal!(caldata)

Progress: 100%|█████████████████████████████████████████| Time: 0:00:01


Iteration: 2, 13244, (189.38966940340094, 369.7930082230989)
Iteration: 3, 29528, (187.858669917108, 360.86351101273874)
Iteration: 4, 30371, (182.49535871754608, 353.94400820695273)
Iteration: 5, 30371, (177.2251824891611, 346.95071659759054)
Iteration Complete!


Dict{String, Array} with 12 entries:
  "nothermf"    => [NaN NaN … NaN NaN; NaN NaN … NaN NaN; … ; NaN NaN … 0.02580…
  "IOF0"        => [0.0168008 0.0168232 … NaN 0.0131417; 0.0142474 0.0148416 … …
  "λ"           => [1553.91, 2352.39, 2701.72, 2282.52, 2591.93]
  "planck1"     => [[1.22365e-34, 6.2563e-34, 3.05786e-33, 1.42051e-32, 6.2291e…
  "photometric" => [NaN NaN … NaN NaN; NaN NaN … NaN NaN; … ; NaN NaN … 0.02580…
  "temp1"       => [NaN NaN … NaN NaN; NaN NaN … NaN NaN; … ; NaN NaN … NaN NaN…
  "idx"         => [104, 184, 219, 177, 208]
  "proj1"       => [0.064606 0.0651608 … NaN 0.0578985; 0.0664853 0.0679098 … 0…
  "tempf"       => [NaN NaN … NaN NaN; NaN NaN … NaN NaN; … ; NaN NaN … NaN NaN…
  "planckf"     => [NaN NaN … NaN NaN; NaN NaN … NaN NaN; … ; NaN NaN … NaN NaN…
  "projf"       => [NaN NaN … NaN NaN; NaN NaN … NaN NaN; … ; NaN NaN … 0.11660…
  "notherm"     => [0.0168008 0.0168232 … NaN 0.0131417; 0.0142474 0.0148416 … …

In [18]:
f = Figure()
image(f[1,1], d["temp1"],interpolate=false)
image(f[1,2], d["tempf"],interpolate=false)
diff = abs.(d["temp1"] .- d["tempf"])
println(count(diff[isfinite.(diff)].>2))
println(diff[125,344])
DataInspector(f)
f

13244
2.882725910785041


In [9]:
xtest = rand(axes(caldata.rdn,1))
ytest = rand(axes(caldata.rdn,2))
# xtest = 243; ytest = 549-293
# xtest = 52; ytest = 7
# xtest = 304; ytest = 340
# xtest=318;ytest=542
# xtest=239;ytest=499
# xtest=388;ytest=165
xtest = 125; ytest = 344

f = Figure()
ax1 = Axis(f[1,1],title="$xtest, $ytest")
ax2 = Axis(f[1,2])
# ax3 = Axis(f[2,1])
# ax4 = Axis(f[2,2])

printfmt("temp1: {:.2f}\ntempf: {:.2f}\n",d["temp1"][xtest,ytest],d["tempf"][xtest,ytest])
println(d["planck1"][xtest,ytest])

lines!(ax1,caldata.wvl,d["IOF0"][xtest,ytest,:])
scatter!(ax1,d["λ"][1:3],[d["IOF0"][xtest,ytest,d["idx"][1]],d["IOF0"][xtest,ytest,d["idx"][2]],d["proj1"][xtest,ytest]])
lines!(ax1,d["λ"][1:3],[d["IOF0"][xtest,ytest,d["idx"][1]],d["IOF0"][xtest,ytest,d["idx"][2]],d["proj1"][xtest,ytest]],color=:red)

lines!(ax2,caldata.wvl,d["IOF0"][xtest,ytest,:])
lines!(ax2,caldata.wvl,d["planck1"][xtest,ytest])
lines!(ax2,caldata.wvl,d["notherm"][xtest,ytest,:])
lines!(ax2,caldata.wvl,d["photometric"][xtest,ytest,:])
lines!(ax2,caldata.wvl,d["nothermf"][xtest,ytest,:])

# lines!(ax3,caldata.wvl,d["photometric"][xtest,ytest,:])
# scatter!(ax3,[d["λ"][4],d["λ"][5],d["λ"][3]],[d["photometric"][xtest,ytest,d["idx"][4]],d["photometric"][xtest,ytest,d["idx"][5]],d["projf"][xtest,ytest]])
# lines!(ax3,[d["λ"][4],d["λ"][5],d["λ"][3]],[d["photometric"][xtest,ytest,d["idx"][4]],d["photometric"][xtest,ytest,d["idx"][5]],d["projf"][xtest,ytest]],color=:red)

# lines!(ax4,caldata.wvl,d["photometric"][xtest,ytest,:])
# lines!(ax4,caldata.wvl,d["planckf"][xtest,ytest,:])
# lines!(ax4,caldata.wvl,d["nothermf"][xtest,ytest,:])

f

temp1: 313.52
tempf: 306.82
[1.3182750621617574e-31, 5.913134538744334e-31, 2.547698315581513e-30, 1.0480248912560434e-29, 4.087083073626501e-29, 1.5041229319007765e-28, 5.4091765092646715e-28, 1.853965834420614e-27, 6.1131245607874716e-27, 1.978261546830694e-26, 6.022608852884261e-26, 1.7710781179607138e-25, 5.072299479258178e-25, 1.4228703237735501e-24, 3.7279478846915664e-24, 9.646689699701742e-24, 2.4431401488551366e-23, 6.031796764029614e-23, 1.4540543345130475e-22, 3.4325832990014838e-22, 7.843787899255897e-22, 1.753512303901412e-21, 3.833855002158923e-21, 8.158505470087241e-21, 1.7172210903947748e-20, 3.529713290912596e-20, 7.121937861909766e-20, 1.423035752591197e-19, 2.774073505172806e-19, 5.340948086141712e-19, 1.0120151014566453e-18, 1.883511271267398e-18, 3.4895485158037374e-18, 6.4463913488787354e-18, 1.139197090379443e-17, 1.970867552369234e-17, 3.3971337541877494e-17, 5.801537873037724e-17, 9.86551770581948e-17, 1.65062369380337e-16, 2.7521422865733968e-16, 4.46623009392

In [80]:
f = Figure()
image(f[1,1],d["temp1"][:,end:-1:1])
f